# 02 — Patch Extraction & Augmentation Visualisation

- Show the sliding-window patch grid on a sample image
- Visualise augmented patch pairs (image + mask)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from data.dataset import CrackDataset
from data.transforms import get_train_transforms

In [ ]:
# Load dataset (without augmentation for patch-grid visualisation)
ds_raw = CrackDataset(
    '../concreteCrackSegmentationDataset',
    '../data/splits/train.txt',
    patch_size=512,
    overlap=128,
)
print(f'Total patches: {len(ds_raw)}')

In [ ]:
# Show patch grid for the first image
from PIL import Image
rgb_p, _ = ds_raw.image_pairs[0]
img = Image.open(rgb_p)
W, H = img.size

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
patch_size, stride = 512, 384
for y in range(0, H, stride):
    for x in range(0, W, stride):
        rect = mpatches.Rectangle((x, y), patch_size, patch_size,
                                   linewidth=1, edgecolor='red', facecolor='none', alpha=0.5)
        ax.add_patch(rect)
ax.set_title(f'{rgb_p.name}  ({W}×{H})  — patch grid (512px, stride 384px)')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Augmented patch examples
ds_aug = CrackDataset(
    '../concreteCrackSegmentationDataset',
    '../data/splits/train.txt',
    transform=get_train_transforms(),
)

fig, axes = plt.subplots(4, 2, figsize=(10, 16))
for i in range(4):
    img_t, mask_t = ds_aug[i * 10]
    # Denormalise for display
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_np = img_t.permute(1, 2, 0).numpy() * std + mean
    img_np = np.clip(img_np, 0, 1)
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'Augmented patch {i}')
    axes[i, 1].imshow(mask_t.squeeze().numpy(), cmap='gray')
    axes[i, 1].set_title('Mask')
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('Augmented training patches', fontsize=14)
plt.tight_layout()
plt.show()